# M5b · Backfill `fact_telemetry_daily`

**Goal:** aggregate `staging.archive` pings into per-(tenant, device, date)
rows. Compresses the 55M-row archive into a few-hundred-thousand-row table
suitable for monthly mart aggregation.

**SQL file:** `sql/16_fact_telemetry_daily_incr.sql`.
Bound parameters: `:window_start`, `:window_end`, `:etl_run_id`,
`:ping_seconds` (default 30), `:rpm_high_threshold` (default 3000).

**What lands per row:** observation_count, ignition_on_minutes,
moving_minutes, idle_minutes, idle_ratio, avg/max speed,
avg/max RPM, high_rpm_seconds, total_fuel_used.

**Exit criteria:**
- One row per active device per day (modulo data gaps).
- `idle_ratio` distributed in [0, 1] — no NaNs after COALESCE.
- Watermark advances to last archive day.

> **Run order:** can run in parallel with `06_load_fact_harsh_event` —
> the two facts read the same staging table but write to different fact
> tables and never contend on the same rows.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs

In [ ]:
from accent_fleet.config import load_pipeline_config
from accent_fleet.db.sql_loader import load_sql
from datetime import datetime, timedelta

cfg = load_pipeline_config()
CHUNK_DAYS = cfg['window']['backfill_chunk_days']
MIN_TIME   = datetime.fromisoformat(cfg['window']['min_event_time'].replace('Z','+00:00')).replace(tzinfo=None)
TEL = cfg.get('archive_telemetry', {})
print('chunk_days =', CHUNK_DAYS, '  min_time =', MIN_TIME)
print('telemetry params =', TEL)

with get_engine().connect() as conn:
    end_time = conn.execute(text("SELECT MAX(date) FROM staging.archive")).scalar_one()
print('staging.archive max event-time =', end_time)

print('\n----- SQL file preview -----')
print(load_sql('16_fact_telemetry_daily_incr.sql')[:800], '...')


## 3. Execute

In [ ]:
import importlib, time, pandas as pd
import accent_fleet.transforms.facts as facts_module
from accent_fleet.pipeline.run_log import begin_run, end_run

facts_module = importlib.reload(facts_module)
load_fact_incremental = facts_module.load_fact_incremental

# Daily aggregation is heavy on GROUP BY — keep chunks small.
TEL_CHUNK_DAYS = 7

run_id = begin_run(mode='notebook:07_load_fact_telemetry_daily')
print('etl_run_id =', run_id)

progress = []
cursor = MIN_TIME
t0 = time.time()
try:
    while cursor < end_time:
        next_cursor = min(cursor + timedelta(days=TEL_CHUNK_DAYS), end_time)
        res = load_fact_incremental('fact_telemetry_daily', run_id=run_id, window_end=next_cursor)
        progress.append({'window_start': cursor, 'window_end': next_cursor,
                         'rows_loaded': res.rows_loaded})
        cursor = next_cursor
    end_run(run_id, status='success', rows_loaded=sum(p['rows_loaded'] for p in progress))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc))
    raise

print(f'done in {time.time()-t0:.1f}s — {len(progress)} chunks')
pd.DataFrame(progress).tail(10)


## 4. Inspect

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    wm = pd.read_sql(text("SELECT * FROM warehouse.etl_watermark WHERE table_name='fact_telemetry_daily'"), conn)
    total = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_telemetry_daily')).scalar_one()
    distros = pd.read_sql(text('''
        SELECT
          PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY idle_ratio)         AS p50_idle,
          PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY idle_ratio)         AS p95_idle,
          AVG(observation_count)                                            AS avg_obs_per_day,
          AVG(ignition_on_minutes)                                          AS avg_ignition_min,
          AVG(moving_minutes)                                               AS avg_moving_min
        FROM warehouse.fact_telemetry_daily
        WHERE idle_ratio IS NOT NULL
    '''), conn)
    sample = pd.read_sql(text('''
        SELECT * FROM warehouse.fact_telemetry_daily
        ORDER BY telemetry_date DESC LIMIT 5
    '''), conn)

print(f'fact_telemetry_daily total rows: {total:,}')
display(wm); display(distros); display(sample)
assert wm['last_event_time'].iloc[0] is not None, 'watermark did not advance'
print('OK — fact_telemetry_daily backfill complete.')
